# День 8: регуляризация — dropout, шум на входе, weight decay (~1.5 ч на модель)
Итог day7: GRU-192 не лучше GRU-128 на маске (0.66259 против 0.66370), а усреднение эпох gru_full не помогло. Главное из логов: **обе модели переобучаются** — валидация лучшая на эпохе 5 из 8, дальше падает, а лосс на train продолжает снижаться (у GRU-192 с 0.67 до 0.63). Сейчас регуляризации практически нет (weight decay 1e-4 за всё обучение почти ничего не делает).

Что меняется в обучении (на сабмит не влияет: при предсказании dropout и шум выключены, numba-движок тот же):
- `--dropout 0.2` — на входе GRU, между слоями GRU и на входе головы;
- `--input-noise 0.05` — гауссов шум к нормированным признакам;
- `--wd 1e-2` — weight decay только на матрицах (не на смещениях).

Запуски:
1. **reg192** — GRU-192 с регуляризацией (~1.5 ч). Сравнение: gru192 без неё — 0.66259.
2. **reg128** — GRU-128 с той же регуляризацией (~1.5 ч, можно в следующей сессии). Сравнение: gru_full — 0.66370.

После каждого запуска можно сразу выполнить шаги 6–8: оценка, выбор лучшего сабмита, проверка, скачивание.

Перед запуском: **T4 GPU**; загрузите на Drive вместо `wunder/day2` папку `хакатон/day2` — изменились `model.py`, `train.py`.

## 1. GPU + Drive + код + numba как у скорера

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx 2>&1 | grep -v "dependency resolver\|requires numba\|incompatible" || true
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!grep -q "input-noise" train.py && echo "code: OK (regularization options present)" || echo "OLD CODE on Drive - upload the new day2 folder"

## 2. Данные: train.parquet из стартового пакета (~6 мин)

In [ ]:
import glob
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=30 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*datasets/train.parquet' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
print(TRAIN_PQ)

## 3. Валидация — те же 600 последовательностей `valid_sub` (~1.5 мин)

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid

## 4. reg192: GRU-192 с регуляризацией (~1.5 ч)
Если сессия оборвалась: ячейки 1–3, затем эта же ячейка с `--resume` в конце команды.

In [ ]:
REG = '--dropout 0.2 --input-noise 0.05 --wd 1e-2'
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $W/runs/reg192 \
    --arch gru --hidden 192 --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs $REG

## 5. reg128: GRU-128 с той же регуляризацией (~1.5 ч; можно пропустить и вернуться позже)

In [ ]:
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $W/runs/reg128 \
    --arch gru --hidden 128 --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs $REG

## 6. Все модели на одной валидации (~10 мин)

In [ ]:
import os
CANDS = {
    'ft_swa':   '+'.join(W + f'/runs/ft_baseline_full/epoch_{k}.pt' for k in range(2, 6)),
    'gru_full': W + '/runs/gru_full/best.pt',
    'gru192':   W + '/runs/gru192_full/best.pt',
    'reg192':   W + '/runs/reg192/best.pt',
    'reg128':   W + '/runs/reg128/best.pt',
}
FOUND = {k: v for k, v in CANDS.items() if all(os.path.exists(p) for p in v.split('+'))}
print('models:', list(FOUND), '| missing:', [k for k in CANDS if k not in FOUND])
MODELS = ' '.join(f'--model {k}={v}' for k, v in FOUND.items())
!python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds8 $MODELS --report $W/day8.json

## 7. Выбор сабмита
Кандидаты — ft_swa плюс одна из наших GRU (пара) или плюс две GRU размера 128 (тройка; модели 192 в тройку не берутся — не влезет по времени). Сборка — только если смесь лучше отправленной (ft_swa+gru_full, 0.66987) хотя бы на 0.0003.

| состав | время от принятой GRU |
|---|---|
| ft_swa + GRU-128 | 0.53x |
| ft_swa + GRU-192 | 0.79x |
| ft_swa + две GRU-128 | 0.80x |

In [ ]:
import json
eb = json.load(open(W + '/day8.json'))['equal_blends']
SENT = 'ft_swa+gru_full'
BIG = {'gru192', 'reg192'}
def speed(key):
    parts = key.split('+')[1:]
    return '0.80x' if len(parts) == 2 else ('0.79x' if parts[0] in BIG else '0.53x')
cands = sorted(((v, k) for k, v in eb.items()
                if k.startswith('ft_swa+') and (k.count('+') == 1 or (k.count('+') == 2 and not BIG & set(k.split('+'))))),
               reverse=True)
print(f'sent: {SENT} {eb[SENT]:.5f}')
for v, k in cands[:6]:
    print(f'  {k:32s} {v:.5f} ({v - eb[SENT]:+.5f})  time {speed(k)}')
BEST_V, BEST = cands[0]
BUILD = BEST_V > eb[SENT] + 0.0003
OUT = W + '/submissions/ens8_' + BEST.replace('+', '_')
SPEC = ' '.join(f'--model "{n}={FOUND[n]}"' for n in BEST.split('+'))
REF = W + '/submissions/gru128x2_v1'
print('build:', BEST if BUILD else 'nothing - no candidate beats the sent ensemble by 0.0003')

## 8. Сборка, проверка, скорость, скачивание (~10 мин)

In [ ]:
if BUILD:
    !python build_ensemble.py --valid-dir /content/data/valid --out $OUT $SPEC
    !python check_submission.py {OUT}.zip --data $W/colab_data/valid_sub.parquet --seqs 3 --rows 20000 --ref-dir $REF
    !python bench_speed.py $REF $OUT --rows 5000 --repeats 5
    from google.colab import files
    files.download(OUT + '.zip')

## Что прислать
Лог обучения (строки `VALID` по эпохам) и вывод ячеек 6–8. Если zip собрался и проверка прошла — можно отправлять.